In [ ]:
from import_dados import *
from notebooks.data_cleasing_sinistros import cria_coluna_acidente_fatal, trata_hora
import polars_ds as pds
from polars import col as c

In [ ]:
sinistros = read_sinistros_raw().pipe(cria_coluna_acidente_fatal).pipe()

# Considerações iniciais

Para a análise estamos considerando apenas os registros ocorridos em vias públicas e que tenham um tipo de sinistro definido segundo as regras no dicionário de dados disponibilizado.
Sendo assim, os registros com tp_sinistro_primario = 'NAO DISPONIVEL' não serão considerados

O agrupamento do tipo de sinistro primário é
definido seguindo a priorização (de 1 a 14)
dos seus respectivos tipos de sinistro:
- ATROPELAMENTO: 1 - pedestre, 2 - vítima
fora do veículo;
- COLISAO: 3 - frontal, 4 - traseira,
5 - lateral, 6 - transversal, 7 - outros;
- CHOQUE: 8;
- OUTROS: 9 - capotamento, 10 -
engavetamento, 11 - tombamento, 12 -
atropelamento animal, 13 - outros;

# Primeira análise

Analisando os dados dos sinistros, encontramos registros com o tipo de via 'NAO DISPONIVEL'   
Todos os registros dessa categoria são sinalizados como fatais, sendo um dado que não reflete a realidade e distorce a associação entre tipo de via e fatalidade. Denotando um viés de seleção para a categoria  
A representatividade dos registros é muito baixa, com apenas 0,08% do total, porém tem um impacto grande na análise de Qui quadrado.  
A remoção dos registros "NAO DISPONIVEL" reduz o Qui quadrado em aproximadamente 33.9% (de 11574 para 6833)
Portanto a retirada destes registros deve ser realizada dos registros.

In [3]:
result_total = sinistros \
.select(pds.chi2("tipo_via", "acidente_fatal").alias("teste_chi2")) \
.unnest("teste_chi2")

result_filtrado = sinistros \
.filter(c('tipo_via') != 'NAO DISPONIVEL') \
.select(pds.chi2("tipo_via", "acidente_fatal").alias("teste_chi2")) \
.unnest("teste_chi2")

print(result_total)
print(result_filtrado)

sinistros = sinistros.filter(c('tipo_via') != 'NAO DISPONIVEL')

shape: (1, 2)
┌──────────────┬────────┐
│ statistic    ┆ pvalue │
│ ---          ┆ ---    │
│ f64          ┆ f64    │
╞══════════════╪════════╡
│ 34928.213836 ┆ 0.0    │
└──────────────┴────────┘
shape: (1, 2)
┌─────────────┬────────┐
│ statistic   ┆ pvalue │
│ ---         ┆ ---    │
│ f64         ┆ f64    │
╞═════════════╪════════╡
│ 8476.762354 ┆ 0.0    │
└─────────────┴────────┘


In [4]:
sinistros.group_by('dia_da_semana', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .plot.bar(x='dia_da_semana', 
              y=alt.Y('acidentes').stack(None), 
              row='acidente_fatal', 
              column='tipo_via') \
    .properties(height=150, width=350) \
    #.resolve_scale(y='independent')
    

alt.Chart(...)

In [8]:
sinistros = sinistros.pipe(trata_hora)

In [9]:
sinistros.group_by('hora', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .with_columns(c('hora').cast(pl.String).str.zfill(2)) \
    .plot.bar(x='hora', 
              y=alt.Y('acidentes').stack(None), 
              column='acidente_fatal', 
              row='tipo_via') \
    .properties(height=150, width=350) \
    .resolve_scale(y='independent')

alt.Chart(...)

In [10]:
sinistros.filter(c('tipo_via') != 'NAO DISPONIVEL') \
    .filter(c('hora').is_not_null()) \
    .group_by('dia_da_semana', 'hora', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .with_columns(c('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y='dia_da_semana', row='tipo_via', column='acidente_fatal') \
    .properties(height=150, width=500) \
    .resolve_scale(color='independent')


alt.Chart(...)

In [11]:
sinistros.filter(c('tipo_via') != 'NAO DISPONIVEL') \
    .filter(c('hora').is_not_null()) \
    .filter(c('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'hora', 'tipo_via').agg(pl.len().alias('acidentes')) \
    .with_columns(c('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y='dia_da_semana', row='tipo_via') \
    .properties(height=150, width=500)

alt.Chart(...)

In [13]:
sinistros.with_columns((pl.when(c('acidente_fatal') == 'SIM').then(1).otherwise(0)).alias('acidente_fatal')) \
    .group_by('dia_da_semana', 'hora', 'tipo_via') \
    .agg(
        (pl.sum('acidente_fatal') *100 / pl.count('acidente_fatal')).alias('taxa_de_fatalidade')
    ) \
    .with_columns(c('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='taxa_de_fatalidade', y='dia_da_semana', row='tipo_via').properties(height=150, width=550) \
        .resolve_scale(color='independent')


alt.Chart(...)

In [14]:

sinistros.filter(c('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'hora', 'tipo_via', 'tp_sinistro_primario').agg(pl.len().alias('acidentes')) \
    .with_columns(c('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y='dia_da_semana', column='tipo_via', row='tp_sinistro_primario') \
    .properties(height=150, width=500)

alt.Chart(...)